In [ ]:
# ==========================================================
# CAPSTONE PREPROCESSING PIPELINE
# Using Existing Dataset Variable: df
# ==========================================================

# Import Libraries
import pandas as pd
import numpy as np

# Preprocessing
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# ==========================================================
# STEP 1: COPY ORIGINAL DATA
# ==========================================================

# Preserve original dataset
data = df.copy()

# ==========================================================
# STEP 2: REMOVE DUPLICATES
# ==========================================================

data = data.drop_duplicates()

# ==========================================================
# STEP 3: REMOVE TARGET COLUMN
# ==========================================================
# Since clustering is unsupervised

X = data.drop('class', axis=1)

# ==========================================================
# STEP 3.5: OUTLIER FLAGGING (NO REMOVAL)
# ==========================================================

# IQR method for outlier detection
Q1 = X.quantile(0.25)
Q3 = X.quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Flag rows with any outlier in any feature
X['outlier_flag'] = ((X < lower_bound) | (X > upper_bound)).any(axis=1)

# ==========================================================
# STEP 4: IDENTIFY NUMERIC FEATURES
# ==========================================================

numeric_features = X.columns.tolist()

# ==========================================================
# STEP 5: PREPROCESSING PIPELINE
# ==========================================================

numeric_pipeline = Pipeline(steps=[

    # Fill Missing Values
    ('imputer', SimpleImputer(strategy='median')),

    # Standardize Data
    ('scaler', StandardScaler())

])

# Column Transformer
preprocessor = ColumnTransformer(transformers=[

    ('num', numeric_pipeline, numeric_features)

])

# ==========================================================
# STEP 6: CREATE NON-PCA DATASET
# ==========================================================

X_no_pca = preprocessor.fit_transform(X)

# Convert to DataFrame
df_no_pca = pd.DataFrame(
    X_no_pca,
    columns=numeric_features
)

# ==========================================================
# STEP 7: CREATE PCA DATASET
# ==========================================================

pca = PCA(n_components=2)

X_with_pca = pca.fit_transform(X_no_pca)

# PCA DataFrame
df_with_pca = pd.DataFrame(
    X_with_pca,
    columns=['PC1', 'PC2']
)

# ==========================================================
# STEP 8: STORE EXPLAINED VARIANCE
# ==========================================================

explained_variance = pca.explained_variance_ratio_

variance_summary = pd.DataFrame({
    'Principal_Component': ['PC1', 'PC2'],
    'Explained_Variance_Ratio': explained_variance
})

# ==========================================================
# STEP 9: SAVE READY-TO-USE DATASETS
# ==========================================================

df_no_pca.to_csv('banknote_preprocessed_no_pca.csv', index=False)
df_with_pca.to_csv('banknote_preprocessed_with_pca.csv', index=False)

# ==========================================================
# STEP 10: PRINT SUMMARY
# ==========================================================

print("="*50)
print("PREPROCESSING SUMMARY")
print("="*50)

print(f"\nOriginal Shape: {df.shape}")
print(f"After Cleaning: {data.shape}")

print("\nFeatures Used:")
print(numeric_features)

print("\nNo PCA Dataset Shape:")
print(df_no_pca.shape)

print("\nPCA Dataset Shape:")
print(df_with_pca.shape)

print("\nExplained Variance Ratio:")
print(variance_summary)

print("\nTotal Variance Retained:")
print(explained_variance.sum())

print("\nDatasets Saved Successfully.")